In [3]:
import pandas as pd
import requests
import numpy as np
from datetime import datetime
import time
import os
from pathlib import Path

In [74]:
url = "https://transparency.entsog.eu/api/v1/operatorpointdirections"

params = {
    "limit": -1,
    "tSOCountry": "BE",
    "hasData": 1,
}

resp = requests.get(url, params=params)
df = pd.DataFrame(resp.json()["operatorpointdirections"])
df["pointId"] = (df["operatorKey"] + df["pointKey"] + df["directionKey"]).str.lower()
df.to_csv(rf"C:\Users\PC\Documents\Projets\TFE\Datas\Gaz\Operator_Directions\Operator_point_directions_belgium.csv")

In [ ]:
url = "https://transparency.entsog.eu/api/v1/operationaldatas"

operator_points = pd.read_csv(rf"C:\Users\PC\Documents\Projets\TFE\Datas\Gaz\Operator_Directions\Operator_point_directions_belgium.csv")
all_year = np.arange(2021,datetime.now().year+1)
metriques_tot = [] 

for i in range (len(all_year)):
    metriques = [] 
    year = all_year[i]
    path_year = rf'C:\Users\PC\Documents\Projets\TFE\Datas\Gaz\Operational_datas\Belgium\{year}'
    if not os.path.exists(path_year): os.makedirs(path_year)

    for j in range(len(operator_points)):
        point_name = operator_points["pointKey"][j] + '_' + operator_points["directionKey"][j]
        point_id = operator_points["pointId"][j]
        from_date = f"{year}-01-01"
        if year == datetime.now().year:
            to_date = datetime.now().strftime("%Y-%m-%d")
        else : 
            to_date = f"{year}-12-31"

        parameters = {
        "indicator": "Physical Flow",
        "pointDirection": point_id,
        "from": from_date,
        "to": to_date,
        "periodType": "hour",
        "timezone": "CET",
        "limit": -1,
        }
        debut = time.perf_counter()
        try:
            reponse = requests.get(url, params=parameters)
            reponse.raise_for_status()
        except requests.exceptions.RequestException as e:
            metriques.append({"annee": year, "point": point_name, "lignes": 0,
                                              "taille_ko": 0, "duree_s": 0})
            continue

        df = pd.DataFrame(reponse.json()["operationaldatas"])
        fichier = f"{path_year}/{point_name}.csv"
        if df.shape[1] == 35:
            df.to_csv(fichier, index=False)

            duree = time.perf_counter() - debut
            taille_ko = os.path.getsize(fichier) / 1024

            metriques.append({"annee": year, "point": point_name, "lignes": len(df),
                    "taille_ko": taille_ko, "duree_s": duree})
            metriques_tot.append({"annee": year, "point": point_name, "lignes": len(df),
                            "taille_ko": taille_ko, "duree_s": duree})
    pd.DataFrame(metriques).to_csv(f"{path_year}/metriques_ingestion.csv", index=False)
pd.DataFrame(metriques_tot).to_csv(f"{path_year}/metriques_belgium_ingestion.csv", index=False)

Récupérer des données sur GIE (ALSI + AGSI) pour avoir la capacité de stockage et LNG terminals

In [4]:
key = "dcb05c571cf3516f36271125d54c9e42"

In [5]:
operators = requests.get("https://agsi.gie.eu/api/about?show=listing",
                       headers={"x-key": key}).json()
country = []
for operator in operators:
    if operator['country'] not in country:
        country.append(operator['country'])

In [6]:
country

['AT',
 'BE',
 'BG',
 'HR',
 'CZ',
 'DK',
 'FR',
 'DE',
 'HU',
 'IE',
 'IT',
 'LV',
 'NL',
 'PL',
 'PT',
 'RO',
 'SK',
 'ES',
 'SE',
 'UA',
 'GB*',
 'GB']

In [7]:
headers = {"x-key": key}

params = {"country": "BE", "size": 300, "page": 1}


BE_gaz_storage = requests.get("https://agsi.gie.eu/api", headers=headers, params=params).json()

###
###   Dans la doc, on peut récupérer que 300 lignes par 300 ligens, il est conseillé d'utilisé la paramètre "page" afin de récupérer page par page les données historiques
###

print("Nombre de pages à récupérer :", BE_gaz_storage ["last_page"])
df_total_data_BE = pd.DataFrame()

for page_number in range(1,BE_gaz_storage["last_page"] + 1):

    params["page"] = page_number
    df_data_paged = pd.DataFrame(requests.get("https://agsi.gie.eu/api", headers=headers, params=params).json()["data"])

    df_total_data_BE = pd.concat([df_total_data_BE,df_data_paged], ignore_index=True)


df_total_data_BE.to_csv(rf"C:\Users\PC\Documents\Projets\TFE\Datas\Gaz\Capacity\Belgium\gaz_storage_capacity_BE.csv")


Nombre de à récupérer : 20


In [8]:
params = {"country": "BE", "size": 300, "page": 1}

BE_lng = requests.get("https://alsi.gie.eu/api", headers=headers, params=params).json()
print("Nombre de pages à récupérer :", BE_lng["last_page"])

df_total_lng_BE = pd.DataFrame()
for page_number in range(1, BE_lng["last_page"] + 1):
    params["page"] = page_number
    df_data_paged = pd.DataFrame(requests.get("https://alsi.gie.eu/api", headers=headers, params=params).json()["data"])

    df_total_lng_BE = pd.concat([df_total_lng_BE, df_data_paged], ignore_index=True)

df_total_lng_BE.to_csv(rf"C:\Users\PC\Documents\Projets\TFE\Datas\Gaz\Capacity\Belgium\LNG_capacity_BE.csv")


Nombre de pages à récupérer : 18
